# Lab 4 — Build the Audience, Then Buy It
Name: Donkihlova Feofila | Group: КМ-43

## Setup

In [24]:
!pip install -q kagglehub

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")

programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")

survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)

acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

print("Loaded successfully from:", DATA)
print("survey:", survey.shape, "| survey_layout:", survey_layout.shape)
print("viewer_attributes cols:", list(viewer_attributes.columns))
print("daily_weights cols:", list(daily_weights.columns))

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.
Loaded successfully from: /kaggle/input/adtech-lab4-tv-audiences
survey: (18349, 3) | survey_layout: (12, 6)
viewer_attributes cols: ['respondent_id', 'attr_id', 'value']
daily_weights cols: ['respondent_id', 'resp_type', 'broadcast_date', 'weight']


In [25]:
print("survey_layout")
display(survey_layout)

print("\nsurvey: VALUE & KEYWORD")
print("VALUE:", survey["VALUE"].value_counts(dropna=False).to_dict())
print("KEYWORD:", survey["KEYWORD"].value_counts(dropna=False).to_dict())
print("\nSample survey rows:")
display(survey.head(10))

print("\nattributes")
display(attributes)

print("\ndaily_weights resp_type")
print(daily_weights["resp_type"].value_counts(dropna=False).to_dict())
print("Sample respondent_id in daily_weights:", daily_weights["respondent_id"].head(5).tolist())

survey_layout


,HIERARCHY,RECTYPE,KEYWORD,Text,Variable,SAMPLE
0,1,H,H7100,Athletic shoes,APPAREL > ATHLETIC SHOES,NaN
1,1.1,H,H7101,Pairs bought in last 12 months,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,NaN
2,1.1.1,B,X710100,Any,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,4779
3,1.1.1,B,X710101,1 pair,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1810
4,1.1.1,B,X710102,2 pairs,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1308
5,1.1.1,B,X710103,3 or more pairs,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1661
6,1.2,H,H7102,Brands bought in last 12 months,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,NaN
7,1.2.1,B,X710201,STRIDE,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,1563
8,1.2.1,B,X710202,KESTREL,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,354
9,1.2.1,B,X710203,EASYWALK,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,1050



survey: VALUE & KEYWORD
VALUE: {'1': 16410, '1.0': 262, 'y': 246, 'TRUE': 241, '0': 237, 'Y': 228, 'yes': 217, '0.0': 129, 'N': 127, 'no': 126, '': 124, 'VALUE': 2}
KEYWORD: {'X710100': 4582, 'X710298': 2804, 'X710101': 2021, 'X710103': 1801, 'X710201': 1734, 'X710102': 1509, 'X710203': 1216, 'X710202': 516, 'X710299': 366, 'x710100': 210, 'X880140': 187, 'X902001': 176, 'X880141': 169, 'X880112': 161, 'x710298': 110, 'X710100 ': 109, 'x710101': 76, 'x710103': 73, 'x710201': 71, 'x710102': 67, 'x710203': 58, 'X710298 ': 54, 'X710103 ': 41, 'X710101 ': 40, 'X710102 ': 31, 'X710201 ': 29, 'X710203 ': 27, 'x710202': 27, 'X710299 ': 14, 'x710299': 11, 'X710202 ': 11, 'x902001': 10, 'x880112': 8, 'x880141': 7, 'x880140': 6, 'X880112 ': 5, 'X880141 ': 4, 'X902001 ': 3, 'KEYWORD': 2, 'X880140 ': 2, '': 1}

Sample survey rows:


,RESPID,KEYWORD,VALUE
0,10415501,X710298,1
1,10072201,x710100,1
2,10302802,X710299,1
3,10070002,X710100,1
4,10130702,X710203,1
5,10243701,X710100,1
6,10342601,X710103,0
7,10399201,X710298,1
8,10364102,X710298,1
9,10200102,X710102,1



attributes


,attr_id,attr_group,attr_name,value_type,applies_to
0,101,Respondent type,R: panel member (household member),flag,"H,R,L,S"
1,102,Respondent type,L: long-term visitor,flag,"H,R,L,S"
2,103,Respondent type,"S: short-term visitor (guest); id ends 81, 82,...",flag,"H,R,L,S"
3,104,Respondent type,H: household record (id ending 00),flag,"H,R,L,S"
4,201,Sex,Male,flag,"R,L,S"
5,202,Sex,Female,flag,"R,L,S"
6,300,Age,Age in years,numeric,"R,L"
7,302,Age band,2-5,flag,"R,L,S"
8,306,Age band,6-11,flag,"R,L,S"
9,312,Age band,12-17,flag,"R,L,S"



daily_weights resp_type
{'R': 284872, 'H': 120232, 'L': 2457}
Sample respondent_id in daily_weights: [10000100, 10000101, 10000102, 10000300, 10000301]


## Task 1.0 — Clean the survey file

In [26]:
raw_rows = len(survey)
rl_ids = set(viewer_attributes.loc[viewer_attributes["attr_id"].isin([101, 102]), "respondent_id"])
ages = viewer_attributes.loc[viewer_attributes["attr_id"] == 300, ["respondent_id", "value"]].copy()
ages["age"] = pd.to_numeric(ages["value"], errors="coerce")
adult_ids = set(ages.loc[ages["respondent_id"].isin(rl_ids) & (ages["age"] >= 18), "respondent_id"].astype(str))

layout_b = survey_layout[survey_layout["RECTYPE"] == "B"].copy()
layout_b["SAMPLE"] = layout_b["SAMPLE"].astype(int)
valid_keywords = set(layout_b["KEYWORD"].str.strip())

df = survey.copy()
bad_records = []

is_header = (df["RESPID"].str.strip() == "RESPID") | (df["KEYWORD"].str.strip() == "KEYWORD")
if is_header.sum() > 0:
    bad_records.append(("Repeated header row", int(is_header.sum()), "Dropped"))
    df = df[~is_header].copy()

is_blank = (df["RESPID"].str.strip() == "") | (df["KEYWORD"].str.strip() == "")
if is_blank.sum() > 0:
    bad_records.append(("Blank / missing RESPID or KEYWORD", int(is_blank.sum()), "Dropped"))
    df = df[~is_blank].copy()

kw_fixed = df["KEYWORD"].str.strip().str.upper()
n_kw_format = (kw_fixed != df["KEYWORD"]).sum()
if n_kw_format > 0:
    bad_records.append(("Lowercase or whitespace in KEYWORD", int(n_kw_format), "Stripped and uppercased"))
df["KEYWORD"] = kw_fixed

id_stripped = df["RESPID"].str.strip()
id_no_zeros = id_stripped.str.lstrip("0")
n_id_zeros = (id_no_zeros != df["RESPID"]).sum()
if n_id_zeros > 0:
    bad_records.append(("Whitespace or leading zeros in RESPID", int(n_id_zeros), "Stripped whitespace/leading zeros"))

id_fixed = id_no_zeros.str.replace(r"\.0$", "", regex=True)
n_id_dot0 = (id_fixed != id_no_zeros).sum()
if n_id_dot0 > 0:
    bad_records.append(("Trailing '.0' in RESPID (float format)", int(n_id_dot0), "Removed trailing '.0'"))
df["RESPID"] = id_fixed

val_norm = df["VALUE"].str.strip().str.upper()
yes_vals = {"1", "1.0", "Y", "YES", "TRUE"}
is_yes = val_norm.isin(yes_vals)
n_non_yes = (~is_yes).sum()
if n_non_yes > 0:
    bad_records.append(("Non-yes VALUE (0, 0.0, N, no, blank)", int(n_non_yes), "Dropped"))
    df = df[is_yes].copy()

n_alt_yes = (df["VALUE"] != "1").sum()
if n_alt_yes > 0:
    bad_records.append(("Non-standard yes VALUE (1.0, y, Y, yes, TRUE)", int(n_alt_yes), "Normalized to '1'"))
df["VALUE"] = "1"

is_valid_kw = df["KEYWORD"].isin(valid_keywords)
n_unknown_kw = (~is_valid_kw).sum()
if n_unknown_kw > 0:
    bad_records.append(("Unknown KEYWORD (not in layout)", int(n_unknown_kw), "Dropped"))
    df = df[is_valid_kw].copy()

is_adult = df["RESPID"].isin(adult_ids)
n_non_adult = (~is_adult).sum()
if n_non_adult > 0:
    bad_records.append(("Non-adult or unknown RESPID", int(n_non_adult), "Dropped"))
    df = df[is_adult].copy()

n_dups = df.duplicated(subset=["RESPID", "KEYWORD"]).sum()
if n_dups > 0:
    bad_records.append(("Duplicate (RESPID, KEYWORD)", int(n_dups), "Dropped duplicates"))
    df = df.drop_duplicates(subset=["RESPID", "KEYWORD"]).copy()

pair_rank = {"X710101": 1, "X710102": 2, "X710103": 3}
pair_rows = df[df["KEYWORD"].isin(pair_rank)]
pair_counts = pair_rows.groupby("RESPID")["KEYWORD"].nunique()
n_multi_pair_people = int((pair_counts > 1).sum())

if n_multi_pair_people > 0:
    df["_rank"] = df["KEYWORD"].map(pair_rank)
    non_pair = df[df["_rank"].isna()].drop(columns="_rank")
    best_pair = (
        df[df["_rank"].notna()]
        .sort_values("_rank", ascending=False)
        .drop_duplicates(subset=["RESPID"], keep="first")
        .drop(columns="_rank")
    )
    n_dropped_pairs = len(pair_rows) - len(best_pair)
    bad_records.append(("Multiple pair counts for same person", int(n_dropped_pairs), "Kept higher pair count"))
    df = pd.concat([non_pair, best_pair], ignore_index=True)

has_any = set(df.loc[df["KEYWORD"] == "X710100", "RESPID"])
all_buyers = set(df["RESPID"])
missing_any = all_buyers - has_any
if len(missing_any) > 0:
    bad_records.append(("Buyer missing Any (X710100) row", len(missing_any), "Imputed X710100 = 1"))
    imputed = pd.DataFrame({"RESPID": list(missing_any), "KEYWORD": "X710100", "VALUE": "1"})
    df = pd.concat([df, imputed], ignore_index=True)

piv = (
    df.assign(val=1)
    .pivot_table(index="RESPID", columns="KEYWORD", values="val", fill_value=0)
    .reindex(sorted(adult_ids), fill_value=0)
)

panel_survey = pd.DataFrame(index=piv.index)
panel_survey.index.name = "respondent_id"
panel_survey["buyer"] = piv["X710100"] > 0
panel_survey["pairs"] = np.select(
    [piv["X710103"] > 0, piv["X710102"] > 0, piv["X710101"] > 0],
    ["3+", "2", "1"],
    default="0"
)
panel_survey["STRIDE"] = piv["X710201"] > 0
panel_survey["KESTREL"] = piv["X710202"] > 0
panel_survey["EASYWALK"] = piv["X710203"] > 0

print("Survey data rows:", raw_rows)
print("Clean yes answers:", len(df))
print("Adult panel members:", len(adult_ids))
print("Non-buyers:", int((~panel_survey["buyer"]).sum()))
print("People with two pair counts:", n_multi_pair_people)

bad_df = pd.DataFrame(bad_records, columns=["Kind of bad record", "Rows affected", "Action taken"])
display(bad_df)

our_counts = df["KEYWORD"].value_counts().rename("Your count")
check_df = layout_b[["KEYWORD", "Text", "SAMPLE"]].merge(
    our_counts, left_on="KEYWORD", right_index=True, how="left"
)
check_df["Your count"] = check_df["Your count"].fillna(0).astype(int)
check_df["Diff"] = check_df["Your count"] - check_df["SAMPLE"]
display(check_df)

raw_kestrel = survey.loc[survey["KEYWORD"] == "X710202", "RESPID"].nunique()
clean_kestrel = int(panel_survey["KESTREL"].sum())
raw_buyers = survey.loc[survey["KEYWORD"] == "X710100", "RESPID"].nunique()
clean_buyers = int(panel_survey["buyer"].sum())
print(f"\nRaw vs Clean Kestrel count: {raw_kestrel} vs {clean_kestrel} (diff: {raw_kestrel - clean_kestrel:+d}, {(raw_kestrel/clean_kestrel - 1)*100:+.1f}%)")
print(f"Raw vs Clean buyer share: {raw_buyers/len(adult_ids):.3f} vs {clean_buyers/len(adult_ids):.3f} (diff: {(raw_buyers - clean_buyers)/len(adult_ids):+.3f})")

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,Kind of bad record,Rows affected,Action taken
0,Repeated header row,2,Dropped
1,Blank / missing RESPID or KEYWORD,1,Dropped
2,Lowercase or whitespace in KEYWORD,1104,Stripped and uppercased
3,Whitespace or leading zeros in RESPID,922,Stripped whitespace/leading zeros
4,Trailing '.0' in RESPID (float format),533,Removed trailing '.0'
5,"Non-yes VALUE (0, 0.0, N, no, blank)",742,Dropped
6,"Non-standard yes VALUE (1.0, y, Y, yes, TRUE)",1194,Normalized to '1'
7,Unknown KEYWORD (not in layout),738,Dropped
8,Non-adult or unknown RESPID,423,Dropped
9,"Duplicate (RESPID, KEYWORD)",933,Dropped duplicates


,KEYWORD,Text,SAMPLE,Your count,Diff
2,X710100,Any,4779,4779,0
3,X710101,1 pair,1810,1810,0
4,X710102,2 pairs,1308,1308,0
5,X710103,3 or more pairs,1661,1661,0
7,X710201,STRIDE,1563,1563,0
8,X710202,KESTREL,354,354,0
9,X710203,EASYWALK,1050,1050,0
10,X710298,OTHER BRAND,2739,2739,0
11,X710299,DON'T KNOW,371,371,0



Raw vs Clean Kestrel count: 495 vs 354 (diff: +141, +39.8%)
Raw vs Clean buyer share: 0.548 vs 0.597 (diff: -0.048)


Answer: The raw survey file broke the vendor's specification in several ways: repeated header rows (2), a blank row (1), inconsistent casing and trailing spaces in KEYWORD (1,104), leading zeros/whitespace (922) and float .0 suffixes (533) in RESPID, non-yes values such as 0/N/no/blank (742) alongside non-standard yes formats (1,194), unknown keywords outside the layout (738), non-adult or unknown respondent IDs (423), duplicate (RESPID, KEYWORD) pairs (933), respondents with two pair counts (113, resolved by keeping the higher count), and buyers missing the Any (X710100) flag (238, imputed). If used as delivered without cleaning, the Kestrel buyer count would have been 495 instead of 354 (+141, or +39.8% too high), while the buyer share among adults would have been 0.548 instead of 0.597 (-0.048, or 4.8 percentage points too low).

## Task 1.1 — Four definitions

In [27]:
sex_df = viewer_attributes.loc[viewer_attributes["attr_id"].isin([201, 202]), ["respondent_id", "attr_id"]].copy()
sex_df["sex"] = sex_df["attr_id"].map({201: "Men", 202: "Women"})
sex_df["respondent_id"] = sex_df["respondent_id"].astype(str)

age_df = viewer_attributes.loc[viewer_attributes["attr_id"] == 300, ["respondent_id", "value"]].copy()
age_df["age"] = pd.to_numeric(age_df["value"], errors="coerce")
age_df["respondent_id"] = age_df["respondent_id"].astype(str)

adults_meta = (
    panel_survey.reset_index()
    .merge(age_df[["respondent_id", "age"]], on="respondent_id", how="left")
    .merge(sex_df[["respondent_id", "sex"]], on="respondent_id", how="left")
)

adults_meta["age_group"] = pd.cut(
    adults_meta["age"], bins=[17, 34, 54, 150], labels=["18-34", "35-54", "55+"]
).astype(str)
adults_meta["demo_group"] = adults_meta["sex"] + " " + adults_meta["age_group"]

adults_meta["Adults 18+"] = True
adults_meta["Shoe buyers"] = adults_meta["buyer"]
adults_meta["Heavy buyers (3+)"] = adults_meta["pairs"] == "3+"
adults_meta["Kestrel buyers"] = adults_meta["KESTREL"]
adults_meta["Adults 25-54"] = (adults_meta["age"] >= 25) & (adults_meta["age"] <= 54)

dw_adults = daily_weights[
    daily_weights["resp_type"].isin(["R", "L"])
].copy()
dw_adults["respondent_id"] = dw_adults["respondent_id"].astype(str)
dw_adults = dw_adults.merge(adults_meta, on="respondent_id", how="inner")

n_days = dw_adults["broadcast_date"].nunique()
assert n_days == 31, f"Expected 31 broadcast days, got {n_days}"

audiences = ["Adults 18+", "Shoe buyers", "Heavy buyers (3+)", "Kestrel buyers", "Adults 25-54"]

aud_sizes = {}
for aud in audiences:
    daily_sum = dw_adults.loc[dw_adults[aud]].groupby("broadcast_date")["weight"].sum()
    aud_sizes[aud] = daily_sum.reindex(dw_adults["broadcast_date"].unique(), fill_value=0).mean()

print("Adults 18+ (M):", round(aud_sizes["Adults 18+"] / 1e6, 1))
print("Shoe buyers (M):", round(aud_sizes["Shoe buyers"] / 1e6, 1))
print("Heavy buyers, 3+ pairs (M):", round(aud_sizes["Heavy buyers (3+)"] / 1e6, 1))
print("Kestrel buyers (M):", round(aud_sizes["Kestrel buyers"] / 1e6, 1))
print("Adults 25-54 (M):", round(aud_sizes["Adults 25-54"] / 1e6, 1))
print("Kestrel buyers in panel (n):", int(adults_meta["Kestrel buyers"].sum()))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [28]:
groups = ["Men 18-34", "Men 35-54", "Men 55+", "Women 18-34", "Women 35-54", "Women 55+"]

shares = pd.DataFrame(index=groups, columns=audiences, dtype=float)
for aud in audiences:
    sub = dw_adults.loc[dw_adults[aud]]
    grp_daily = sub.groupby(["broadcast_date", "demo_group"])["weight"].sum().unstack(fill_value=0)
    grp_mean = grp_daily.reindex(columns=groups, fill_value=0).mean()
    shares[aud] = grp_mean / aud_sizes[aud]

indices = shares.div(shares["Adults 18+"], axis=0) * 100

summary_1_2 = pd.concat(
    {"Share": shares.round(3), "Index": indices.round(1)}, axis=1
).swaplevel(axis=1).sort_index(axis=1, level=0)

ordered_cols = [(aud, metric) for aud in audiences for metric in ["Share", "Index"]]
summary_1_2 = summary_1_2[ordered_cols]
display(summary_1_2)

Adults 18+        Shoe buyers        Heavy buyers (3+)         \
                 Share  Index       Share  Index             Share  Index   
Men 18-34        0.144  100.0       0.140   97.4             0.111   77.1   
Men 35-54        0.160  100.0       0.158   99.2             0.138   86.8   
Men 55+          0.185  100.0       0.186  100.8             0.189  102.2   
Women 18-34      0.144  100.0       0.140   97.6             0.139   97.0   
Women 35-54      0.160  100.0       0.178  111.4             0.226  141.9   
Women 55+        0.208  100.0       0.197   94.6             0.196   94.0   

            Kestrel buyers        Adults 25-54         
                     Share  Index        Share  Index  
Men 18-34            0.118   82.1        0.174  121.2  
Men 35-54            0.189  118.6        0.326  204.1  
Men 55+              0.126   68.2        0.000    0.0  
Women 18-34          0.181  126.0        0.174  121.2  
Women 35-54          0.265  165.8        0.326  204.1  
Women 55+            0.121   58.0        0.000    0.0

Answer: I would hand the planner Heavy buyers (3+ pairs) 49.6M adults, 1,661 panelists. It directly matches Kestrel's brief, has enough panel sample size for reliable program-level ratings unlike Kestrel buyers, which has only 354 panelists / 10.0M people and only targets existing customers rather than category buyers, and avoids the waste of Adults 25–54 which excludes 55+ heavy buyers who represent 38.5% of the segment. "Athletic-shoe buyers" looks almost identical to all adults indices between 94.6 and 111.4 because nearly 60% of US adults 148.1M of 252.6M buy at least one pair of athletic shoes a year—it is too broad a mass-market category to differentiate audiences. Finally, because the purchase attribute was fused onto the panel using only demographics. Age, sex, and household income, it cannot tell you whether a program's viewers have a true behavioral affinity for athletic shoes beyond their demographics (e.g., whether sports or fitness-show viewers buy more running shoes than other people of the exact same age, sex, and income).

## Task 2.1 — Programs by audience

In [30]:
prime_oct = programs[programs["daypart"] == "Prime"]
for kw in ["NFL", "Dancing", "NCIS", "Hannity", "Unsellable", "Halloween", "Grotesquerie"]:
    m = prime_oct[prime_oct["program_name"].str.contains(kw, case=False, na=False)]
    print(kw, "->", m[["network_code", "network_name", "program_name"]].drop_duplicates().values.tolist())

NFL -> [['ABC', 'ABC', 'NFL Football'], ['NBC', 'NBC', 'NFL Football']]
Dancing -> [['ABC', 'ABC', 'Dancing with the Stars']]
NCIS -> [['CBS', 'CBS', 'NCIS'], ['CBS', 'CBS', 'NCIS: Origins']]
Hannity -> [['FNC', 'Fox News Channel', 'Hannity']]
Unsellable -> [['HGTV', 'HGTV', 'Unsellable Houses']]
Halloween -> [['FOOD', 'Food Network', 'Halloween Wars'], ['FOOD', 'Food Network', 'Halloween Baking Championship']]
Grotesquerie -> [['FX', 'FX', 'Grotesquerie']]


In [31]:
progs = programs.copy()
progs["start_ts"] = pd.to_datetime(progs["start_ts"])
progs["end_ts"] = pd.to_datetime(progs["end_ts"])
progs["broadcast_date"] = pd.to_datetime(progs["broadcast_date"]).dt.strftime("%Y-%m-%d")

views = viewings[viewings["resp_type"].isin(["R", "L"])].copy()
views["respondent_id"] = views["respondent_id"].astype(str)
views["start_ts"] = pd.to_datetime(views["start_ts"])
views["end_ts"] = pd.to_datetime(views["end_ts"])

aud_flags = adults_meta[["respondent_id", "Adults 18+", "Heavy buyers (3+)", "Kestrel buyers"]].copy()
views_adults = views.merge(aud_flags, on="respondent_id", how="inner")

dw_dates = dw_adults.copy()
dw_dates["broadcast_date"] = pd.to_datetime(dw_dates["broadcast_date"]).dt.strftime("%Y-%m-%d")
daily_aud_size = {}
for aud in ["Adults 18+", "Heavy buyers (3+)", "Kestrel buyers"]:
    daily_aud_size[aud] = (
        dw_dates.loc[dw_dates[aud]]
        .groupby("broadcast_date")["weight"]
        .sum()
        .to_dict()
    )

rc_prog = rate_card[rate_card["program_name"].notna()].copy()
rc_time = rate_card[rate_card["program_name"].isna()].copy()

def _to_min(hhmm):
    h, m = map(int, str(hhmm).split(":"))
    return h * 60 + m

rc_time["start_min"] = rc_time["start_time"].map(_to_min)
rc_time["end_min"] = rc_time["end_time"].map(_to_min)

def get_telecast_cost(row):
    net = row["network_code"]
    pname = row["program_name"]
    m_prog = rc_prog[(rc_prog["network_code"] == net) & (rc_prog["program_name"] == pname)]
    if len(m_prog) > 0:
        return float(m_prog["cost_30s_usd"].iloc[0])
    wd = row["start_ts"].weekday()
    t_min = row["start_ts"].hour * 60 + row["start_ts"].minute
    cand = rc_time[rc_time["network_code"] == net]
    for _, r in cand.iterrows():
        wds = list(r["weekdays"]) if r["weekdays"] is not None else []
        if wd not in wds:
            continue
        s, e = r["start_min"], r["end_min"]
        if s < e:
            if s <= t_min < e:
                return float(r["cost_30s_usd"])
        else:  # перехід через північ
            if t_min >= s or t_min < e:
                return float(r["cost_30s_usd"])
    return np.nan

def score_telecasts(telecasts_df):
    rows = []
    views_by_net = {net: g for net, g in views_adults.groupby("network_code")}

    for _, tc in telecasts_df.iterrows():
        net = tc["network_code"]
        t_start, t_end = tc["start_ts"], tc["end_ts"]
        dur = float(tc["duration_min"])
        bdate = tc["broadcast_date"]
        cost = get_telecast_cost(tc)

        vnet = views_by_net.get(net)
        if vnet is None:
            ov_df = pd.DataFrame()
        else:
            ov = vnet[(vnet["start_ts"] < t_end) & (vnet["end_ts"] > t_start)].copy()
            if len(ov) > 0:
                s = np.maximum(ov["start_ts"].values, np.datetime64(t_start))
                e = np.minimum(ov["end_ts"].values, np.datetime64(t_end))
                ov["ov_min"] = (e - s) / np.timedelta64(1, "m")
                ov["w_min"] = ov["weight"] * ov["ov_min"]
            ov_df = ov

        rec = {
            "telecast_id": tc["telecast_id"],
            "broadcast_date": bdate,
            "cost": cost,
        }
        for aud, col in [("Adults 18+", "adults"), ("Heavy buyers (3+)", "heavy"), ("Kestrel buyers", "kestrel")]:
            if len(ov_df) > 0:
                avg_viewers = ov_df.loc[ov_df[aud], "w_min"].sum() / dur
            else:
                avg_viewers = 0.0
            rec[f"{col}_viewers"] = avg_viewers
            rec[f"{col}_rating"] = (avg_viewers / daily_aud_size[aud][bdate]) * 100.0
        rows.append(rec)
    return pd.DataFrame(rows)

target_programs = [
    ("NFL Football", "NBC", "NFL Football (NBC)"),
    ("NFL Football", "ABC", "NFL Football (ABC)"),
    ("Dancing with the Stars", "ABC", "Dancing with the Stars (ABC)"),
    ("NCIS", "CBS", "NCIS (CBS)"),
    ("Hannity", "FNC", "Hannity (Fox News)"),
    ("Unsellable Houses", "HGTV", "Unsellable Houses (HGTV)"),
    ("Halloween Wars", "FOOD", "Halloween Wars (Food Network)"),
    ("Grotesquerie", "FX", "Grotesquerie (FX)"),
]

prime_oct = progs[progs["daypart"] == "Prime"].copy()

prog_results = []
for pname, ncode, label in target_programs:
    sub = prime_oct[(prime_oct["program_name"] == pname) & (prime_oct["network_code"] == ncode)]
    tc_scores = score_telecasts(sub)

    adults_rtg = tc_scores["adults_rating"].mean()
    target_rtg = tc_scores["heavy_rating"].mean()
    kestrel_rtg = tc_scores["kestrel_rating"].mean()

    adults_viewers = tc_scores["adults_viewers"].mean()
    target_viewers = tc_scores["heavy_viewers"].mean()

    comp = target_viewers / adults_viewers
    idx = (target_rtg / adults_rtg) * 100.0
    kestrel_idx = (kestrel_rtg / adults_rtg) * 100.0
    cost = tc_scores["cost"].mean()
    adults_cpm = cost / adults_viewers * 1000.0
    target_cpm = cost / target_viewers * 1000.0

    prog_results.append({
        "Program": label,
        "Telecasts": len(sub),
        "Adults rating": round(adults_rtg, 2),
        "Target rating": round(target_rtg, 2),
        "Target viewers (k)": round(target_viewers / 1000.0, 1),
        "Composition": round(comp, 3),
        "Index": round(idx, 1),
        "Cost :30 ($)": round(cost),
        "Adults CPM ($)": round(adults_cpm, 2),
        "Target CPM ($)": round(target_cpm, 2),
        "Kestrel index": round(kestrel_idx, 1),
        "_cost_exact": cost,
        "_target_viewers_exact": target_viewers,
    })

task2_df = pd.DataFrame(prog_results)

cols_2_1 = [
    "Program", "Adults rating", "Target rating", "Target viewers (k)",
    "Composition", "Index", "Cost :30 ($)", "Adults CPM ($)", "Target CPM ($)"
]
display(task2_df[cols_2_1])
print("Target universe (M):", round(aud_sizes["Heavy buyers (3+)"] / 1e6, 1))

,Program,Adults rating,Target rating,Target viewers (k),Composition,Index,Cost :30 ($),Adults CPM ($),Target CPM ($)
0,NFL Football (NBC),5.82,5.79,2865.0,0.195,99.3,487200,33.12,170.05
1,NFL Football (ABC),8.93,8.36,4149.1,0.184,93.6,723600,32.09,174.40
2,Dancing with the Stars (ABC),3.25,3.14,1564.2,0.191,96.8,158600,19.33,101.39
3,NCIS (CBS),2.21,2.43,1206.3,0.216,110.1,178800,32.09,148.22
4,Hannity (Fox News),1.09,0.84,416.9,0.152,77.1,39900,14.50,95.71
5,Unsellable Houses (HGTV),0.50,0.64,317.2,0.249,126.8,12900,10.12,40.66
6,Halloween Wars (Food Network),0.28,0.33,161.9,0.228,116.2,9700,13.67,59.90
7,Grotesquerie (FX),0.26,0.25,124.2,0.188,95.9,7592,11.50,61.11


Target universe (M): 49.6


## Task 2.2 — Index, volume or price?

In [32]:
highest_idx_prog = task2_df.loc[task2_df["Index"].idxmax(), "Program"]
most_viewers_prog = task2_df.loc[task2_df["Target viewers (k)"].idxmax(), "Program"]
lowest_cpm_prog = task2_df.loc[task2_df["Target CPM ($)"].idxmin(), "Program"]

print("Highest index (program):", highest_idx_prog)
print("Most target viewers (program):", most_viewers_prog)
print("Lowest target CPM (program):", lowest_cpm_prog)

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): NFL Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


Answer: They are not the same program: Unsellable Houses (HGTV) has both the highest index (126.8) and the lowest target CPM ($40.66), while NFL Football (ABC) delivers the most target viewers (4,149.1k — over 13 times more per airing than Unsellable Houses). To a client asking for "only the highest index," I would explain (as in Lecture 5's shampoo example) that index measures concentration, not scale. High-index cable shows have small audiences (0.64 rating, 317.2k viewers) and limited weekly inventory, so buying only high-index programs caps your reach and misses the vast majority of heavy buyers watching big broadcast shows. A smart buy balances high-efficiency cable with high-reach broadcast programs.

## Task 2.3 — Kestrel buyers

In [33]:
display(task2_df[["Program", "Index", "Kestrel index"]].rename(columns={"Index": "Heavy-buyer index"}))

,Program,Heavy-buyer index,Kestrel index
0,NFL Football (NBC),99.3,98.5
1,NFL Football (ABC),93.6,85.9
2,Dancing with the Stars (ABC),96.8,82.7
3,NCIS (CBS),110.1,56.5
4,Hannity (Fox News),77.1,42.6
5,Unsellable Houses (HGTV),126.8,76.9
6,Halloween Wars (Food Network),116.2,95.2
7,Grotesquerie (FX),95.9,129.6


Answer: The Kestrel index swings wildly (from 42.6 on Hannity to 129.6 on Grotesquerie, and drops to 76.9 on Unsellable Houses) because of tiny sample size: there are only 354 Kestrel buyers in the entire panel (versus 1,661 heavy buyers). On a low-rated cable program (0.25–0.50 rating), only 1 to 3 Kestrel panelists may watch a given telecast, so a single person tuning in or out shifts the index by dozens of points. I would trust the heavy-buyer index, which has nearly five times the sample size and much lower variance. To check reliability, you can inspect the unweighted count of target panelists watching each program and compute confidence intervals (or bootstrap standard errors across panelists/telecasts) around each index.

In [34]:
for name, df in [
    ("acr_sessions", acr_sessions),
    ("hh_attributes", hh_attributes),
    ("reference_ratings", reference_ratings),
    ("universe", universe),
]:
    print(f"=== {name} (shape={df.shape}) ===")
    print("cols:", list(df.columns))
    display(df.head(3))

=== acr_sessions (shape=(6433473, 9)) ===
cols: ['session_id', 'tv_id', 'hashed_ip', 'session_type', 'network_code', 'input_source', 'start_utc', 'end_utc', 'dma_code']


,session_id,tv_id,hashed_ip,session_type,network_code,input_source,start_utc,end_utc,dma_code
0,900000001,7000360,59ef26c76b07e8d2,live,MTV,cable_box,2024-10-14 10:00:00+00:00,2024-10-14 10:06:27+00:00,641
1,900000002,7000382,189ea07d1cd85aea,live,ESPN,cable_box,2024-10-14 10:00:00+00:00,2024-10-14 10:04:21+00:00,602
2,900000003,7000449,488e94fe8965aa77,live,TOON,satellite_box,2024-10-14 10:00:00+00:00,2024-10-14 10:19:26+00:00,532


=== hh_attributes (shape=(54276, 8)) ===
cols: ['household_id', 'hashed_ip', 'region', 'income_band', 'head_age_band', 'hh_size', 'children', 'new_car_intender_6mo']


,household_id,hashed_ip,region,income_band,head_age_band,hh_size,children,new_car_intender_6mo
0,91349081,e572b9dc47d706f0,South,$100k-$150k,50-64,1,False,0.068
1,11015405,5ea8356d60dc046f,South,$25k-$50k,18-34,2,False,0.083
2,18473717,c284381637c51f4e,South,$100k-$150k,35-49,1,False,0.094


=== reference_ratings (shape=(8, 2)) ===
cols: ['telecast_id', 'hh_rating']


,telecast_id,hh_rating
0,1013636,25.86
1,1021823,33.14
2,1004042,10.56


=== universe (shape=(12, 3)) ===
cols: ['head_age', 'income_band', 'tv_households']


,head_age,income_band,tv_households
0,65+,$100k-$150k,4914000
1,65+,$150k+,5050000
2,65+,$25k-$50k,7791000


## Task 3.1 — Weights

In [35]:
tv_ips = acr_sessions[["tv_id", "hashed_ip"]].drop_duplicates()
tv_hh = tv_ips.merge(hh_attributes[["household_id", "hashed_ip", "head_age_band", "income_band"]], on="hashed_ip", how="inner")

hh_per_tv = tv_hh.groupby("tv_id")["household_id"].nunique()
valid_tv_ids = hh_per_tv[hh_per_tv == 1].index

matched_tvs = (
    tv_hh[tv_hh["tv_id"].isin(valid_tv_ids)]
    .drop_duplicates(subset=["tv_id"])
    .copy()
)
matched_tvs["head_age"] = np.where(matched_tvs["head_age_band"] == "65+", "65+", "Under 65")

matched_tvs = matched_tvs.merge(universe, on=["head_age", "income_band"], how="left")
print("TVs matched to one household:", len(matched_tvs))

acr_m = acr_sessions[acr_sessions["tv_id"].isin(valid_tv_ids)].copy()
acr_m["start_et"] = pd.to_datetime(acr_m["start_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr_m["end_et"] = pd.to_datetime(acr_m["end_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr_m["broadcast_date"] = (acr_m["start_et"] - pd.Timedelta(hours=6)).dt.strftime("%Y-%m-%d")

in_tab = acr_m[["tv_id", "broadcast_date"]].drop_duplicates()
in_tab = in_tab.merge(matched_tvs[["tv_id", "head_age", "income_band", "tv_households"]], on="tv_id", how="left")

cell_counts = (
    in_tab.groupby(["broadcast_date", "head_age", "income_band"])
    .agg(in_tab_tvs=("tv_id", "count"), tv_households=("tv_households", "first"))
    .reset_index()
)
cell_counts["weight"] = cell_counts["tv_households"] / cell_counts["in_tab_tvs"]

in_tab = in_tab.merge(
    cell_counts[["broadcast_date", "head_age", "income_band", "weight"]],
    on=["broadcast_date", "head_age", "income_band"],
    how="left"
)

daily_weight_sum = in_tab.groupby("broadcast_date")["weight"].sum()
print("Weights sum per day (M):", round(daily_weight_sum.mean() / 1e6, 1))

ref_tcs = reference_ratings.merge(
    progs[["telecast_id", "network_code", "network_name", "program_name", "broadcast_date", "start_ts", "end_ts"]],
    on="telecast_id",
    how="left"
)

ref_nets = set(ref_tcs["network_code"])
acr_live = acr_m[
    (acr_m["session_type"] == "live") & (acr_m["network_code"].isin(ref_nets))
].copy()
acr_live = acr_live[(acr_live["end_et"] - acr_live["start_et"]).dt.total_seconds() >= 60]

rows_3 = []
for _, tc in ref_tcs.iterrows():
    bdate = tc["broadcast_date"]
    net = tc["network_code"]
    t_start, t_end = tc["start_ts"], tc["end_ts"]
    day_intab = in_tab[in_tab["broadcast_date"] == bdate]
    total_tvs = len(day_intab)
    total_w = day_intab["weight"].sum()
    ov = acr_live[
        (acr_live["network_code"] == net)
        & (acr_live["start_et"] < t_end)
        & (acr_live["end_et"] > t_start)
    ].copy()

    if len(ov) > 0:
        s = np.maximum(ov["start_et"].values, np.datetime64(t_start))
        e = np.minimum(ov["end_et"].values, np.datetime64(t_end))
        ov_sec = (e - s) / np.timedelta64(1, "s")
        watched_ids = set(ov.loc[ov_sec >= 60, "tv_id"])
    else:
        watched_ids = set()

    watched_df = day_intab[day_intab["tv_id"].isin(watched_ids)]

    unw_rtg = len(watched_df) / total_tvs * 100.0
    w_rtg = watched_df["weight"].sum() / total_w * 100.0
    ref_rtg = float(tc["hh_rating"])

    rows_3.append({
        "Telecast": f"{tc['program_name']} ({tc['network_code']}, {bdate})",
        "Unweighted": round(unw_rtg, 2),
        "Weighted": round(w_rtg, 2),
        "Reference": round(ref_rtg, 2),
        "Gap unweighted": abs(unw_rtg - ref_rtg),
        "Gap weighted": abs(w_rtg - ref_rtg),
    })

task3_df = pd.DataFrame(rows_3)
display(task3_df[["Telecast", "Unweighted", "Weighted", "Reference"]])

mean_gap_unw = task3_df["Gap unweighted"].mean()
mean_gap_w = task3_df["Gap weighted"].mean()
share_closed = (mean_gap_unw - mean_gap_w) / mean_gap_unw

print("Mean gap unweighted (points):", round(mean_gap_unw, 2))
print("Mean gap weighted (points):", round(mean_gap_w, 2))
print("Share of gap closed:", round(share_closed, 3))

TVs matched to one household: 19607
Weights sum per day (M): 125.8


,Telecast,Unweighted,Weighted,Reference
0,"MLB Postseason (FOX, 2024-10-25)",17.03,17.76,25.86
1,"NFL Football (NBC, 2024-10-27)",22.62,23.75,33.14
2,"NCIS (CBS, 2024-10-14)",5.50,5.80,10.56
3,"Hannity (FNC, 2024-10-16)",1.04,1.35,2.30
4,"Unsellable Houses (HGTV, 2024-10-16)",2.12,2.08,3.69
5,"Halloween Baking Championship (FOOD, 2024-10-14)",0.95,0.90,1.63
6,"Grotesquerie (FX, 2024-10-16)",2.17,2.11,3.53
7,"The Real Housewives of New York City (BRAVO, 2...",0.71,0.64,1.23


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


## Task 3.2 — What weighting cannot fix

Answer: Weighting moved ratings up toward the reference for programs with older (65+) audiences—most notably NFL Football (NBC) (+1.13 pts), MLB Postseason (FOX) (+0.73 pts), Hannity (FNC) (+0.31 pts), and NCIS (CBS) (+0.30 pts)—because smart-TV footprints under-represent 65+ households, so up-weighting those cells lifts the shows they watch. Conversely, younger-skewing cable shows (The Real Housewives of New York City, Grotesquerie, Halloween Baking Championship, Unsellable Houses) edged slightly down (by 0.04–0.07 pts). Overall, weighting closed only 7.5% of the mean gap (3.73 to 3.44 points) because it cannot fix structural measurement and selection biases from Lab 3 and Lecture 4: a smart-TV ACR footprint measures only a single ACR-enabled TV (missing secondary TVs, out-of-home viewing, and households without smart TVs or opted out of ACR), suffers from ACR detection/matching loss (e.g., unrecognized inputs or streaming), and cannot correct for non-demographic viewing differences within the same age/income cell. Finally, weighting by income barely moves these ratings because prime-time viewing of these broadcast and cable shows varies much more strongly by age than by household income—rebalancing income bands changes the weighted average very little.

## Task 4.1 — October predicts October

In [36]:
dw_p2 = daily_weights[daily_weights["resp_type"].isin(["R", "L"])].copy()
dw_p2["broadcast_date"] = pd.to_datetime(dw_p2["broadcast_date"]).dt.strftime("%Y-%m-%d")
daily_p2_size = dw_p2.groupby("broadcast_date")["weight"].sum().to_dict()
heavy_ids = set(adults_meta.loc[adults_meta["Heavy buyers (3+)"], "respondent_id"])
views_p2 = viewings[viewings["resp_type"].isin(["R", "L"])].copy()
views_p2["respondent_id"] = views_p2["respondent_id"].astype(str)
views_p2["start_ts"] = pd.to_datetime(views_p2["start_ts"])
views_p2["end_ts"] = pd.to_datetime(views_p2["end_ts"])
views_p2["is_heavy"] = views_p2["respondent_id"].isin(heavy_ids)
nets_4 = ["CBS", "NBC", "ABC", "FOX", "FNC"]
prime_4 = progs[
    (progs["daypart"] == "Prime")
    & (progs["network_code"].isin(nets_4))
    & (progs["broadcast_date"] >= "2024-10-01")
    & (progs["broadcast_date"] <= "2024-10-28")
].copy()

def score_p2_and_heavy(telecasts_df):
    views_by_net = {net: g for net, g in views_p2.groupby("network_code")}
    recs = []
    for _, tc in telecasts_df.iterrows():
        net = tc["network_code"]
        t_start, t_end = tc["start_ts"], tc["end_ts"]
        dur = float(tc["duration_min"])
        bdate = tc["broadcast_date"]

        vnet = views_by_net.get(net)
        ov = vnet[(vnet["start_ts"] < t_end) & (vnet["end_ts"] > t_start)].copy()
        if len(ov) > 0:
            s = np.maximum(ov["start_ts"].values, np.datetime64(t_start))
            e = np.minimum(ov["end_ts"].values, np.datetime64(t_end))
            ov["w_min"] = ov["weight"] * ((e - s) / np.timedelta64(1, "m"))
            p2_v = ov["w_min"].sum() / dur
            heavy_v = ov.loc[ov["is_heavy"], "w_min"].sum() / dur
        else:
            p2_v, heavy_v = 0.0, 0.0

        recs.append({
            "telecast_id": tc["telecast_id"],
            "network_code": net,
            "program_name": tc["program_name"],
            "broadcast_date": bdate,
            "p2_actual": (p2_v / daily_p2_size[bdate]) * 100.0,
            "heavy_actual": (heavy_v / daily_aud_size["Heavy buyers (3+)"][bdate]) * 100.0,
        })
    return pd.DataFrame(recs)

tc_4 = score_p2_and_heavy(prime_4)

hist_4 = tc_4[tc_4["broadcast_date"] <= "2024-10-21"]
test_4 = tc_4[tc_4["broadcast_date"] >= "2024-10-22"].copy()
hist_means = (
    hist_4.groupby(["network_code", "program_name"])
    .agg(p2_forecast=("p2_actual", "mean"), heavy_forecast=("heavy_actual", "mean"))
    .reset_index()
)

fc_df = test_4.merge(hist_means, on=["network_code", "program_name"], how="inner")
fc_p2 = fc_df[fc_df["p2_actual"] > 0].copy()
fc_p2["p2_ape"] = (fc_p2["p2_actual"] - fc_p2["p2_forecast"]).abs() / fc_p2["p2_actual"] * 100.0

fc_heavy = fc_df[fc_df["heavy_actual"] > 0].copy()
fc_heavy["heavy_ape"] = (fc_heavy["heavy_actual"] - fc_heavy["heavy_forecast"]).abs() / fc_heavy["heavy_actual"] * 100.0

mape_p2 = fc_p2["p2_ape"].mean()
mape_heavy = fc_heavy["heavy_ape"].mean()

biggest = fc_p2.loc[fc_p2["p2_ape"].idxmax()]
biggest_str = f"{biggest['program_name']} ({biggest['network_code']}), {biggest['broadcast_date']}"

print("Telecasts forecast (n):", len(fc_df))
print("MAPE, P2+ (%):", round(mape_p2, 1))
print("MAPE, heavy buyers (%):", round(mape_heavy, 1))
print("Biggest miss, P2+ (program, date):", biggest_str)

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


## Task 4.2 — Why it missed

In [37]:
top5_misses = (
    fc_p2.sort_values("p2_ape", ascending=False)
    .head(5)
    .assign(Program=lambda d: d["program_name"] + " (" + d["network_code"] + ")")
    [["Program", "broadcast_date", "p2_forecast", "p2_actual", "p2_ape"]]
    .rename(columns={
        "broadcast_date": "Date",
        "p2_forecast": "Forecast",
        "p2_actual": "Actual",
        "p2_ape": "APE (%)"
    })
)
top5_misses["Forecast"] = top5_misses["Forecast"].round(2)
top5_misses["Actual"] = top5_misses["Actual"].round(2)
top5_misses["APE (%)"] = top5_misses["APE (%)"].round(1)
display(top5_misses)

,Program,Date,Forecast,Actual,APE (%)
80,Krapopolis (FOX),2024-10-27,1.06,0.22,387.8
78,Universal Basic Guys (FOX),2024-10-27,0.68,0.26,162.2
37,Fire Country (CBS),2024-10-27,1.70,0.67,154.8
65,Accused (FOX),2024-10-22,0.95,0.37,154.6
4,What Would You Do? (ABC),2024-10-23,0.44,0.18,138.1


Answer: All five biggest misses are sharp over-forecasts—especially on Sunday, October 27 (Krapopolis, Universal Basic Guys on FOX, and Fire Country on CBS), plus Accused (FOX, Oct 22) and What Would You Do? (ABC, Oct 23). Because a simple historical mean ignores schedule context, it misses drops caused by losing a strong NFL doubleheader lead-in (Sunday animations on FOX jump when preceded by late-afternoon NFL games and collapse without one), repeats or out-of-slot airings, and competing live events (e.g., MLB Postseason / World Series). The heavy-buyer forecast is less accurate than P2+ (38.6% vs. 35.8% MAPE) because heavy buyers are a much smaller subset of the panel (1,661 panelists vs. all P2+), so sampling variance is higher—a few panelists tuning in or out causes larger percentage swings on individual telecasts. Two things a better forecast would use are: (1) schedule and content features (lead-in program rating, day-of-week/time-slot, is_repeat, and competing live sports/specials), and (2) recency weighting or shrinkage (giving more weight to recent weeks and shrinking noisy small-sample program means toward network/daypart averages).

## Task 5.1 — The plan

In [38]:
plan_spots = {
    "NFL Football (NBC)": 1,
    "Dancing with the Stars (ABC)": 1,
    "NCIS (CBS)": 1,
    "Unsellable Houses (HGTV)": 10,
    "Halloween Wars (Food Network)": 4,
    "Grotesquerie (FX)": 1,
}

plan_df = task2_df[task2_df["Program"].isin(plan_spots.keys())].copy()
plan_df["Spots"] = plan_df["Program"].map(plan_spots)
plan_df["Cost ($)"] = (plan_df["Spots"] * plan_df["_cost_exact"]).round().astype(int)
plan_df["Target impressions (k)"] = (plan_df["Spots"] * plan_df["_target_viewers_exact"] / 1000.0).round(1)

total_cost = int(plan_df["Cost ($)"].sum())
total_target_imp = (plan_df["Spots"] * plan_df["_target_viewers_exact"]).sum()
total_adults_imp = (plan_df["Spots"] * (plan_df["_target_viewers_exact"] / plan_df["Composition"])).sum()

blended_cpm = total_cost / total_target_imp * 1000.0
blended_comp = total_target_imp / total_adults_imp
waste_share = 1.0 - blended_comp
waste_dollars = total_cost * waste_share

display(plan_df[["Program", "Spots", "Cost ($)", "Target impressions (k)"]])
print("Plan cost ($):", total_cost)
print("Plan target impressions (M):", round(total_target_imp / 1e6, 1))
print(f"Blended target CPM ($): {blended_cpm:.2f}")
print(f"Blended composition: {blended_comp:.3f} (non-target waste: {waste_share*100:.1f}%, ~${waste_dollars:,.0f})")

,Program,Spots,Cost ($),Target impressions (k)
0,NFL Football (NBC),1,487200,2865.0
2,Dancing with the Stars (ABC),1,158600,1564.2
3,NCIS (CBS),1,178800,1206.3
5,Unsellable Houses (HGTV),10,129000,3172.5
6,Halloween Wars (Food Network),4,38800,647.7
7,Grotesquerie (FX),1,7592,124.2


Plan cost ($): 999992
Plan target impressions (M): 9.6
Blended target CPM ($): 104.38
Blended composition: 0.214 (non-target waste: 78.6%, ~$785,722)


## Task 5.2 — Recommendation

Answer: For Kestrel's \$1,000,000 November flight, we recommend spending \$999,992 across a balanced mix of high-reach broadcast and high-efficiency cable—1 spot each on NFL Football (NBC), Dancing with the Stars (ABC), and NCIS (CBS) paired with 10 spots on Unsellable Houses (HGTV), 4 on Halloween Wars (Food Network), and 1 on Grotesquerie (FX)—to deliver 9.6M heavy-buyer impressions at a \$104.38 blended target CPM. The three broadcast spots provide immediate scale (5.6M target impressions in just three airings), while the HGTV and Food Network spots drive efficiency with the highest heavy-buyer indices (126.8 and 116.2) and lowest target CPMs (\$40.66 and \$59.90). We excluded Hannity (weak 77.1 index) and avoided putting the entire budget into cable, where limited weekly inventory would cap reach. Delivery carries meaningful uncertainty: our October backtest showed a 38.6% MAPE for heavy buyers on individual prime-time telecasts (driven by schedule shifts and small-sample variance), and November adds Election-week volatility. To protect the budget against under-delivery, we should negotiate the buy on a guaranteed basis so the networks owe make-goods if actual heavy-buyer impressions fall short of the forecast. Because our plan's blended composition is 0.214, roughly 78.6% of the spend (~$786,000) will reach adults who are not heavy buyers—an expected trade-off for linear TV that still builds awareness among lighter shoe buyers. After the flight, we will audit delivery with a post-buy report comparing actual versus guaranteed heavy-buyer impressions (claiming any owed make-goods), evaluate net reach and frequency, and track brand-lift signals such as Kestrel search volume, website traffic, and retail/DTC sales.